Discover oMEGACat-BH-2

In [ ]:
#import needed packages
from multiprocessing import Pool
import numpy as np
from astropy import units as u
from astropy.table import Table

from periapsis.data.common import AstrometryData
from periapsis.prior.uniform_prior import UniformPrior
from periapsis.prior.normal_prior import NormalPrior
from periapsis.fitting.mcmcCampbell import MCMCCampbell
from periapsis.fitting.ultranestlinear import UltranestLinearFitter
from periapsis.stats import stat_funcs
from periapsis.plotting import plots 


Load in the data table from Whitaker et al. (2026)

In [ ]:
data_table = Table.read("apjlae7a5ct1_mrt.txt",
                format="ascii.mrt")

clean= ~(data_table['Flag'] > 0)

t = np.asarray(data_table['Year'])[clean]
x = np.asarray(data_table['DeltaRA'])[clean]
y = np.asarray(data_table['DeltaDec'])[clean]
x_err = np.asarray(data_table['e_DeltaRA'])[clean]
y_err = np.asarray(data_table['e_DeltaDec'])[clean]
ref_epoch = 2012
m_vis = 0.78 #solar masses



Let's perform our analysis in AU at the distance of Omega Centauri

In [ ]:
dist = 5494 #pc
def mas_to_AU(r):
    return r/1000 * (dist * np.pi)/(180*3600) *206265


x = mas_to_AU(x)
y = mas_to_AU(y)
x_err = mas_to_AU(x_err)
y_err = mas_to_AU(y_err)

Use the package to intialize the data for fitting

In [ ]:
fit_data = AstrometryData(t, x, y, x_err, y_err, ref_epoch)

Create your priors, here we will use the physical Campbell elements to describe the orbit. 

In [ ]:
priors = {
    'P': UniformPrior(1, 200),  # Period in years
    'a': UniformPrior(1, 100),  # Semi-major axis in AU
    'e': UniformPrior(0, 0.95),  # Eccentricity
    'cosi': UniformPrior(-1, 1),  # cos(Inclination) 
    'Omega': UniformPrior(0, 2*np.pi),  # Longitude of ascending node
    'omega': UniformPrior(0, 2*np.pi),  # Argument of periastron 
    't0': UniformPrior(-0.5,0.5), # Time of periastron passage in fraction of period
    'dx': UniformPrior(-10, 10),  # Offset in x (RA) in AU
    'dy': UniformPrior(-10, 10),  # Offset in y (Dec) in AU
    'dpmra': UniformPrior(-10,10),  # Proper motion in RA in AU/yr with a normal prior
    'dpmdec': UniformPrior(-10,10),  # Proper motion in Dec in AU/yr with a normal prior

}

We now have everything needed to get an orbital fit, first we will sample the Campbell elements and the other orbital parameters with mcmc

In [ ]:
with Pool(processes=4) as pool:
    fitter = MCMCCampbell(nwalkers=250, niter=100000, m1=m_vis,m2_max=50, pool=pool, **priors)
    results = fitter.fit(fit_data)

In [ ]:
stats,fit_results = stat_funcs.all_stats(results,data=fit_data)

In [ ]:
plots.all_plots(results=results,data=fit_data,priors=priors,m1=m_vis)

The package offers multiple models and samplers. Here we can try linear least squares with the Thiele Innes coefficients and sample P,e and t0 with Ultranest

In [ ]:
fitter2 = UltranestLinearFitter(nlive=4000,min_ess=1000,m1=m_vis,m2_max=50,**priors)
results2 = fitter2.fit(fit_data)

In [ ]:
stats2,fits_results2 = stat_funcs.all_stats(results2,data=fit_data)

In [ ]:
plots.all_plots(results=results2,data=fit_data,priors=priors,m1=m_vis)